In [1]:
import numpy as np

def collide_1d(m1, m2, u1, u2, e=1.0):
    """Post-collision velocities for a 1D impact with restitution e.

    e = 1  -> perfectly elastic (KE conserved)
    e = 0  -> perfectly inelastic (bodies move together)
    """
    v_cm = (m1*u1 + m2*u2) / (m1 + m2)      # unchanged by the collision
    v1 = v_cm + e * (m2/(m1+m2)) * (u2 - u1)
    v2 = v_cm + e * (m1/(m1+m2)) * (u1 - u2)
    return v1, v2

def p_total(m1, m2, v1, v2):
    return m1*v1 + m2*v2

def ke_total(m1, m2, v1, v2):
    return 0.5*m1*v1**2 + 0.5*m2*v2**2

In [2]:
# ---- Initial values: the three limiting cases, all with e = 1 ----
cases = [
    ("equal masses",   1.0,   1.0, 3.0, 0.0),
    ("light -> heavy", 0.1, 100.0, 5.0, 0.0),
    ("heavy -> light", 100.0, 0.1, 5.0, 0.0),
]

print(f"{'case':<16s}{'v1':>10s}{'v2':>10s}{'dp':>14s}{'dKE':>14s}")
for label, m1, m2, u1, u2 in cases:
    v1, v2 = collide_1d(m1, m2, u1, u2, e=1.0)
    dp  = p_total(m1, m2, v1, v2)  - p_total(m1, m2, u1, u2)
    dke = ke_total(m1, m2, v1, v2) - ke_total(m1, m2, u1, u2)
    print(f"{label:<16s}{v1:10.5f}{v2:10.5f}{dp:14.2e}{dke:14.2e}")

case                    v1        v2            dp           dKE
equal masses       0.00000   3.00000      0.00e+00      0.00e+00
light -> heavy    -4.99001   0.00999      1.11e-16      2.22e-16
heavy -> light     4.99001   9.99001      0.00e+00      2.27e-13


In [3]:
# ---- Initial values: 2 kg block at 4 m/s strikes a stationary 3 kg block ----
m1, m2 = 2.0, 3.0
u1, u2 = 4.0, 0.0

p0  = p_total(m1, m2, u1, u2)
ke0 = ke_total(m1, m2, u1, u2)
print(f"before:  p = {p0:.4f} kg m/s   KE = {ke0:.4f} J")
print(f"v_cm  = {(m1*u1 + m2*u2)/(m1+m2):.4f} m/s  (never changes)\n")

print(f"{'e':>6s}{'v1':>9s}{'v2':>9s}{'p':>10s}{'KE':>10s}{'KE lost':>10s}{'% lost':>9s}")
for e in [1.0, 0.75, 0.5, 0.25, 0.0]:
    v1, v2 = collide_1d(m1, m2, u1, u2, e)
    p  = p_total(m1, m2, v1, v2)
    ke = ke_total(m1, m2, v1, v2)
    print(f"{e:6.2f}{v1:9.4f}{v2:9.4f}{p:10.4f}{ke:10.4f}"
          f"{ke0-ke:10.4f}{100*(1-ke/ke0):8.1f}%")

before:  p = 8.0000 kg m/s   KE = 16.0000 J
v_cm  = 1.6000 m/s  (never changes)

     e       v1       v2         p        KE   KE lost   % lost
  1.00  -0.8000   3.2000    8.0000   16.0000   -0.0000    -0.0%
  0.75  -0.2000   2.8000    8.0000   11.8000    4.2000    26.2%
  0.50   0.4000   2.4000    8.0000    8.8000    7.2000    45.0%
  0.25   1.0000   2.0000    8.0000    7.0000    9.0000    56.2%
  0.00   1.6000   1.6000    8.0000    6.4000    9.6000    60.0%


In [4]:
def resolve_2d(m1, m2, r1, r2, v1, v2, e=1.0):
    """Return post-collision velocities for two circles in contact.

    r1, r2 : position 2-vectors      v1, v2 : velocity 2-vectors
    Returns (v1_new, v2_new, j) where j is the impulse magnitude.
    """
    r1, r2 = np.asarray(r1, float), np.asarray(r2, float)
    v1, v2 = np.asarray(v1, float), np.asarray(v2, float)

    d_vec = r2 - r1
    dist  = np.linalg.norm(d_vec)
    n     = d_vec / dist                 # unit normal, points 1 -> 2

    v_rel = v2 - v1
    v_n   = np.dot(v_rel, n)             # approach speed along the normal

    if v_n > 0:                         # already separating - do NOT resolve
        return v1, v2, 0.0

    j = -(1 + e) * v_n / (1/m1 + 1/m2)
    return v1 - (j/m1)*n, v2 + (j/m2)*n, j


In [5]:
# ---- Initial values: equal masses, glancing (offset) impact ----
m1, m2 = 1.0, 1.0
r1, r2 = [0.0, 0.0], [1.0, 0.5]     # offset in y -> glancing, not head-on
v1, v2 = [3.0, 0.0], [0.0, 0.0]     # body 1 moving +x, body 2 at rest

d_vec = np.array(r2) - np.array(r1)
print(f"separation d = {np.linalg.norm(d_vec):.6f} m")
print(f"unit normal  = {d_vec/np.linalg.norm(d_vec)}\n")

for e in [1.0, 0.5, 0.0]:
    a, b, j = resolve_2d(m1, m2, r1, r2, v1, v2, e)
    p0  = m1*np.array(v1) + m2*np.array(v2)
    p1  = m1*a + m2*b
    ke0 = 0.5*m1*np.dot(v1, v1) + 0.5*m2*np.dot(v2, v2)
    ke1 = 0.5*m1*np.dot(a, a)   + 0.5*m2*np.dot(b, b)
    print(f"e = {e}")
    print(f"   j       = {j:.6f} kg m/s")
    print(f"   v1'     = {a}")
    print(f"   v2'     = {b}")
    print(f"   |dp|    = {np.linalg.norm(p1-p0):.2e}")
    print(f"   KE      = {ke0:.4f} -> {ke1:.4f} J")
    print(f"   v1'.v2' = {np.dot(a, b):.2e}\n")

separation d = 1.118034 m
unit normal  = [0.89442719 0.4472136 ]

e = 1.0
   j       = 2.683282 kg m/s
   v1'     = [ 0.6 -1.2]
   v2'     = [2.4 1.2]
   |dp|    = 0.00e+00
   KE      = 4.5000 -> 4.5000 J
   v1'.v2' = 2.22e-16

e = 0.5
   j       = 2.012461 kg m/s
   v1'     = [ 1.2 -0.9]
   v2'     = [1.8 0.9]
   |dp|    = 0.00e+00
   KE      = 4.5000 -> 3.1500 J
   v1'.v2' = 1.35e+00

e = 0.0
   j       = 1.341641 kg m/s
   v1'     = [ 1.8 -0.6]
   v2'     = [1.2 0.6]
   |dp|    = 0.00e+00
   KE      = 4.5000 -> 2.7000 J
   v1'.v2' = 1.80e+00



In [6]:
def overlapping(b1, b2):
    """Narrow-phase test for two circles. Returns (bool, distance, normal)."""
    d_vec = b2['r'] - b1['r']
    dist  = np.linalg.norm(d_vec)
    if dist == 0.0:                     # exactly coincident - degenerate
        return False, 0.0, np.array([1.0, 0.0])
    return dist < b1['radius'] + b2['radius'], dist, d_vec/dist

In [7]:
def resolve_pair(b1, b2, e):
    """Apply impulse AND separate the overlap. Mutates b1 and b2 in place."""
    hit, dist, n = overlapping(b1, b2)
    if not hit:
        return False

    v_n = np.dot(b2['v'] - b1['v'], n)
    if v_n > 0:                        # separating already
        return False

    # --- 1. velocity response (the physics) ---
    inv_mass = 1/b1['m'] + 1/b2['m']
    j = -(1 + e) * v_n / inv_mass
    b1['v'] = b1['v'] - (j/b1['m']) * n
    b2['v'] = b2['v'] + (j/b2['m']) * n

    # --- 2. positional correction (the bookkeeping) ---
    #     push apart in inverse proportion to mass; heavy bodies barely move
    overlap = b1['radius'] + b2['radius'] - dist
    if overlap > 0:
        b1['r'] = b1['r'] - n * overlap * (1/b1['m']) / inv_mass
        b2['r'] = b2['r'] + n * overlap * (1/b2['m']) / inv_mass
    return True

In [8]:
# ---- Initial values ----
W, H = 10.0, 8.0          # box dimensions (m)
dt      = 0.001            # s
t_max   = 10.0             # s

def make_bodies():
    """Fresh copy of the initial scene - call this to reset."""
    return [
        dict(m=1.0, r=np.array([2.0, 3.0]), v=np.array([ 4.0,  1.5]), radius=0.30),
        dict(m=2.0, r=np.array([6.0, 5.0]), v=np.array([-2.5,  2.0]), radius=0.40),
        dict(m=1.5, r=np.array([4.0, 7.0]), v=np.array([ 1.0, -3.0]), radius=0.35),
        dict(m=3.0, r=np.array([7.5, 2.0]), v=np.array([-1.5, -1.0]), radius=0.50),
    ]

def p_total_2d(bodies):
    return sum(b['m'] * b['v'] for b in bodies)

def ke_total_2d(bodies):
    return sum(0.5 * b['m'] * np.dot(b['v'], b['v']) for b in bodies)

def bounce_walls(b, e):
    """Reflect off the box. NOTE: walls are external - they break momentum."""
    for k, L in ((0, W), (1, H)):
        if b['r'][k] - b['radius'] < 0:
            b['r'][k] = b['radius']
            b['v'][k] = -e * b['v'][k]
        elif b['r'][k] + b['radius'] > L:
            b['r'][k] = L - b['radius']
            b['v'][k] = -e * b['v'][k]

def run(e, walls=True, t_max=t_max, dt=dt):
    bodies = make_bodies()
    p0, ke0 = p_total_2d(bodies).copy(), ke_total_2d(bodies)
    n_collisions = 0

    for _ in range(int(t_max/dt)):
        for b in bodies:                   # 1. move (no forces yet)
            b['r'] = b['r'] + b['v'] * dt
        if walls:                            # 2. walls
            for b in bodies:
                bounce_walls(b, e)
        for i in range(len(bodies)):           # 3. pairwise collisions
            for k in range(i+1, len(bodies)):
                if resolve_pair(bodies[i], bodies[k], e):
                    n_collisions += 1

    return bodies, p0, ke0, n_collisions

In [9]:
# ---- Test 1: isolated system (no walls) - momentum MUST be conserved ----
bodies, p0, ke0, nc = run(e=1.0, walls=False, t_max=3.0)
p, ke = p_total_2d(bodies), ke_total_2d(bodies)
print("NO WALLS, e = 1.0  (isolated system)")
print(f"  collisions : {nc}")
print(f"  p  {p0} -> {p}")
print(f"  |dp|      = {np.linalg.norm(p - p0):.3e} kg m/s")
print(f"  KE {ke0:.6f} -> {ke:.6f} J   (dKE = {ke-ke0:.3e})\n")

# ---- Test 2: with walls, elastic and inelastic ----
for e in [1.0, 0.9]:
    bodies, p0, ke0, nc = run(e=e, walls=True)
    ke = ke_total_2d(bodies)
    print(f"WALLS, e = {e}")
    print(f"  collisions : {nc}")
    print(f"  KE {ke0:.6f} -> {ke:.6f} J   ({100*ke/ke0:.2f}% retained)")

NO WALLS, e = 1.0  (isolated system)
  collisions : 1
  p  [-4. -2.] -> [-4. -2.]
  |dp|      = 9.930e-16 kg m/s
  KE 31.750000 -> 31.750000 J   (dKE = -3.553e-15)

WALLS, e = 1.0
  collisions : 9
  KE 31.750000 -> 31.750000 J   (100.00% retained)
WALLS, e = 0.9
  collisions : 5
  KE 31.750000 -> 13.245001 J   (41.72% retained)
